# Июньский скрипт эквайринга (single month)

## Что делает скрипт
- Формирует `final_df` за один месяц в логике `01_07_acq_dash_scoped_flat_map.ipynb`.
- Сохраняет секционный стиль майской тетрадки: каждый шаг отдельно.
- После расчета применяет апрельские коэффициенты к `commission_monthly`.

## Что нужно заполнить
- `report_month`
- `coef_csv_path`
- `output_csv_path`

## Важные правила
- Логика `commission_monthly` до коэффициентов: `actual -> 08b scoped plan fallback -> legacy`.
- Секция коэффициентов добавляет поля `commission_monthly_lake`, `k_comm_monthly_agr`, `com_forcast`, `coef_source`.
- В тетрадке нет checkpoint, таймеров, probe/heartbeat и лишнего debug-шума.

In [ ]:
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path

import numpy as np
import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))


def normalize_inn(v):
    if pd.isna(v):
        return None
    s = re.sub(r'[^0-9]', '', str(v).strip())
    return s or None


def normalize_inn_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def normalize_agr_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None'}:
        return None
    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass
    s = re.sub(r'\.0$', '', s)
    return s if s not in {'', 'nan', 'None'} else None


def normalize_contract(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    return s if s else None


def to_decimal_or_none(v):
    if pd.isna(v):
        return None
    if isinstance(v, Decimal):
        return v
    try:
        return Decimal(str(v).strip().replace(',', '.'))
    except (InvalidOperation, ValueError):
        return None


MONEY_QUANT = Decimal('0.01')


def to_money_decimal_2_or_none(v):
    d = to_decimal_or_none(v)
    if d is None:
        return None
    try:
        return d.quantize(MONEY_QUANT)
    except InvalidOperation:
        return d


def normalize_text_value(v):
    if pd.isna(v):
        return None
    s = str(v).replace('\xa0', ' ')
    s = re.sub(r'[\t\r\n]+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s or None


def normalize_ssp_ocrm_core(v):
    txt = normalize_text_value(v)
    if txt is None:
        return None

    txt_norm = txt.lower().replace(' ', '')

    if txt_norm.startswith('дкб'):
        return 'ДКБ'
    if txt_norm.startswith('дмсб(ми') or txt_norm.startswith('дммб'):
        return 'ДМ'
    if txt_norm.startswith('дмсб') or txt_norm.startswith('дсб'):
        return 'ДМСБ'
    if txt_norm.startswith('дм'):
        return 'ДМ'
    return None


def normalize_filial_rf(v):
    s = normalize_text_value(v)
    if s is None:
        return None

    m_rf = re.search(r'(?i)рф', s)
    if m_rf:
        s = s[:m_rf.end()]

    s = re.sub(r'(?i)\b(ао|пао|оао|зао)\b.*$', '', s).strip(' ,;.-')
    if not s:
        return None

    s = s.lower()
    s = s[:1].upper() + s[1:]
    s = re.sub(r'(?i)рф', 'РФ', s)
    return s


def _norm_tariff_text_short(v):
    if pd.isna(v):
        return ''
    s = str(v).replace('\xa0', ' ')
    s = re.sub(r'[\t\r\n]+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip().lower()
    return s


def _is_zero_tariff_value(v):
    if pd.isna(v):
        return False
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    return s in {'0', '0.0', '0.00'}


def build_tariff_short(tariff_text, source='lake'):
    txt = _norm_tariff_text_short(tariff_text)

    has_akt = 'акт' in txt
    has_ind = 'индив' in txt
    has_std = 'стандарт' in txt
    has_promo = ('акцион' in txt) or ('меню' in txt) or ('сезон' in txt)
    is_zero_excel = (source == 'excel') and _is_zero_tariff_value(tariff_text)

    if has_akt:
        return 'По актам'
    if has_promo:
        return 'Акционный'
    if has_ind or is_zero_excel:
        return 'Индивидуальный'
    if has_std:
        return 'Стандарт'
    return 'Не сегментирован'


def first_non_empty_value(series):
    for raw_v in series.tolist():
        cleaned = normalize_text_value(raw_v)
        if cleaned is not None:
            return cleaned
    return None


def _split_scope(values, chunk_size):
    scope_values = [str(x).strip() for x in (values or []) if str(x).strip()]
    if not scope_values:
        return []
    safe_chunk_size = max(int(chunk_size or 1), 1)
    return [scope_values[i:i + safe_chunk_size] for i in range(0, len(scope_values), safe_chunk_size)]


def _in_sql_list(values):
    values = [str(x) for x in (values or []) if str(x).strip()]
    return ', '.join([f"'{x}'" for x in values]) if values else "''"


def _run_impala_fetch(imp, sql_text, mem_limit='8g'):
    with imp:
        imp.execute(f'set MEM_LIMIT={mem_limit}')
        return imp.fetch(sql_text)


print('Imports and helpers loaded')

In [ ]:
# Параметры запуска
report_month = '2026-06-01'

coef_csv_path = './кэфы_апрель.csv'
excel_april_compare_path = '/home/jovyan/documents/Equaring/Data/04_Апрель_2026.xlsx'
excel_april_compare_header = 0
output_csv_path = '/home/jovyan/documents/Equaring/Data/final_df_june_2026_with_april_coef.csv'
save_csv = True

run_invalidate_metadata = True
run_refresh_after_invalidate = True

use_fast_flat_08b_map = True
preload_tariff_fix_flat_map = False
use_legacy_commission_monthly_only = False

disable_flatmap_runtime_cache = True
if disable_flatmap_runtime_cache:
    preload_tariff_fix_flat_map = False
    if '_tariff_fix_map_runtime_cache_df' in globals():
        globals().pop('_tariff_fix_map_runtime_cache_df', None)

section09_chunk_threshold = 1800
section09_chunk_size = 800

report_month_ts = pd.to_datetime(report_month)
month_start = report_month_ts.strftime('%Y-%m-%d')
month_end = (report_month_ts + pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')
report_month_label = report_month_ts.strftime('%Y-%m')
snapshot_month_start = month_start

print(f'report_month={report_month_label}, month_start={month_start}, month_end={month_end}')
print(f'commission mode={"actual_then_08b_then_legacy" if not use_legacy_commission_monthly_only else "legacy_only"}')

imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

invalidate_tables = [
    'ods_alpha.scd1_agreements', 'ods_alpha.scd1_companies', 'ods_alpha.scd1_agr_terms',
    'ocrm_ul.s_org_ext', 'cdiul.ext_id_org', 'ods_alpha.scd1_merchants',
    'ods_alpha.scd1_pos_terminals', 'sandbox_ai.shestopalov_terminal_amortization_oneoff',
    'ods_alpha.scd1_trx', 'ods_alpha.scd1_trx_acq', 'ods_alpha.scd1_trx_int',
    'ods_alpha.scd1_base24_fiids', 'ods.scd1_z_r2_ip_merchants', 'ods.scd1_z_r2_tariff_tune',
    'ods.scd1_z_r2_tariff_fix', 'ods.scd1_z_cl_corp', 'ods.scd1_z_depart',
    'ods.scd1_z_branch', 'ods.scd1_z_r2_tariff_plan'
]

if run_invalidate_metadata:
    invalidate_failed = []
    with imp:
        for t in invalidate_tables:
            try:
                imp.execute(f'invalidate metadata {t}')
                if run_refresh_after_invalidate:
                    imp.execute(f'refresh {t}')
            except Exception as exc_inv:
                invalidate_failed.append((t, type(exc_inv).__name__))
    if invalidate_failed:
        print(f'invalidate failures: {len(invalidate_failed)}')

# В scoped-режиме preload обычно выключен; оставляем опцию для совместимости
tariff_fix_map_flat_df = pd.DataFrame(columns=['c_tariff_plan', 'commission_monthly_fix'])
if use_fast_flat_08b_map and preload_tariff_fix_flat_map:
    sql_tariff_fix_map_flat = """
    with tt_pairs as (
      select distinct
        tt.c_tariff_plan as c_tariff_plan_raw,
        tt.c_tariff as c_tariff_raw
      from ods.scd1_z_r2_tariff_tune tt
      where tt.c_tariff_plan is not null
    ),
    tf_agg as (
      select
        tf.id as c_tariff_raw,
        max(cast(tf.c_summa as decimal(18,2))) as commission_monthly_fix
      from ods.scd1_z_r2_tariff_fix tf
      group by tf.id
    )
    select
      cast(p.c_tariff_plan_raw as string) as c_tariff_plan,
      max(a.commission_monthly_fix) as commission_monthly_fix
    from tt_pairs p
    left join tf_agg a
      on p.c_tariff_raw = a.c_tariff_raw
    group by p.c_tariff_plan_raw
    """
    tariff_fix_map_flat_df = _run_impala_fetch(imp, sql_tariff_fix_map_flat, mem_limit='8g')
    if tariff_fix_map_flat_df is None:
        tariff_fix_map_flat_df = pd.DataFrame(columns=['c_tariff_plan', 'commission_monthly_fix'])
    if not tariff_fix_map_flat_df.empty:
        tariff_fix_map_flat_df['c_tariff_plan'] = tariff_fix_map_flat_df['c_tariff_plan'].astype(str).str.strip()
        tariff_fix_map_flat_df['commission_monthly_fix'] = pd.to_numeric(tariff_fix_map_flat_df['commission_monthly_fix'], errors='coerce')
        tariff_fix_map_flat_df = (
            tariff_fix_map_flat_df
            .dropna(subset=['c_tariff_plan'])
            .groupby('c_tariff_plan', as_index=False)['commission_monthly_fix']
            .max()
            .sort_values('c_tariff_plan')
            .reset_index(drop=True)
        )

print(f'preloaded_08b_map_rows={len(tariff_fix_map_flat_df):,}')

## Секция 1. SA-периметр
Кратко: формируем SA-контур договоров на месяц и базовые атрибуты клиента.

In [ ]:
sql_sa_perimeter = f"""
select distinct
  cast(a.n_agr as string) as n_agr,
  cast(a.abs_agr_id as string) as agr_id,
  cast(a.n_cmp_client as string) as n_cmp_client,
  cast(a.c_agr_number as string) as contract_number,
  cast(a.d_valid_from as date) as d_valid_from,
  cast(a.d_valid_to as date) as d_valid_to,
  regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn,
  cast(c.c_cmp_name as string) as company_name
from ods_alpha.scd1_agreements a
join ods_alpha.scd1_companies c
  on c.n_cmp = a.n_cmp_client
where upper(trim(cast(a.acq_class as string))) = 'SA'
  and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
  and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
  and coalesce(a.ods_deleted_flg, '0') <> '1'
  and coalesce(c.ods_deleted_flg, '0') <> '1'
  and c.c_inn is not null
  and exists (
      select 1
      from ods_alpha.scd1_agr_terms t
      where cast(t.n_agr as string) = cast(a.n_agr as string)
        and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
        and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
        and upper(trim(cast(t.cf_ter_type as string))) = 'P'
        and coalesce(t.ods_deleted_flg, '0') <> '1'
  )
"""

sa_df = _run_impala_fetch(imp, sql_sa_perimeter, mem_limit='8g')
if sa_df is None:
    sa_df = pd.DataFrame()
if not sa_df.empty:
    sa_df['inn'] = sa_df['inn'].map(normalize_inn)
    sa_df['contract_number'] = sa_df['contract_number'].map(normalize_contract)

print(f'01_sa_perimeter rows: {len(sa_df):,}')

## Секция 2. CDI map
Кратко: по `inn` подтягиваем `cdi_id` и `ssp_ocrm`.

In [ ]:
inn_values = sorted([
    x for x in sa_df.get('inn', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x
])


def _build_sql_cdi(inn_scope):
    inn_sql_list = _in_sql_list(inn_scope)
    return f"""
    with ocrm_current as (
      select
        regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') as inn,
        cast(soe.row_id as string) as row_id,
        trim(cast(soe.x_area_resp as string)) as x_area_resp_norm,
        trim(cast(soe.x_area_resp as string)) as ssp_ocrm,
        row_number() over (
          partition by regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '')
          order by cast(soe.created as timestamp) desc, cast(soe.row_id as string) desc
        ) as rn
      from ocrm_ul.s_org_ext soe
      where regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') in ({inn_sql_list})
        and coalesce(soe.x_removed_flg, 'N') = 'N'
        and coalesce(soe.x_duplicate_flg, 'N') = 'N'
    ),
    ocrm_one as (
      select inn, row_id, ssp_ocrm, x_area_resp_norm
      from ocrm_current
      where rn = 1
    )
    select
      o.inn,
      o.ssp_ocrm,
      o.x_area_resp_norm,
      cast(e.party_id as string) as cdi_id
    from ocrm_one o
    left join cdiul.ext_id_org e
      on cast(e.cmo_ext_party_source_id as string) = o.row_id
     and upper(cast(e.cmo_ext_source_system as string)) like 'OCRM%'
    """


if not inn_values:
    cdi_map_df = pd.DataFrame(columns=['inn', 'ssp_ocrm_raw', 'ssp_ocrm', 'x_area_resp_norm', 'cdi_id'])
else:
    cdi_map_df = _run_impala_fetch(imp, _build_sql_cdi(inn_values), mem_limit='8g')

if cdi_map_df is None:
    cdi_map_df = pd.DataFrame(columns=['inn', 'ssp_ocrm_raw', 'ssp_ocrm', 'x_area_resp_norm', 'cdi_id'])
if not cdi_map_df.empty:
    cdi_map_df['inn'] = cdi_map_df['inn'].map(normalize_inn)
    cdi_map_df['cdi_id'] = cdi_map_df['cdi_id'].astype(str)
    cdi_map_df['x_area_resp_norm'] = cdi_map_df['x_area_resp_norm'].apply(normalize_text_value)
    cdi_map_df['ssp_ocrm_raw'] = cdi_map_df['x_area_resp_norm']
    cdi_map_df['ssp_ocrm'] = cdi_map_df['ssp_ocrm_raw'].apply(normalize_ssp_ocrm_core)
    cdi_map_df = cdi_map_df.drop_duplicates(subset=['inn'], keep='first')

print(f'02_cdi_map rows: {len(cdi_map_df):,}')

## Секция 3. CFT map
Кратко: связываем `cdi_id` с `cft_id`.

In [ ]:
cdi_values = sorted([
    x for x in cdi_map_df.get('cdi_id', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x
])


def _build_sql_cft(cdi_scope):
    cdi_sql_list = _in_sql_list(cdi_scope)
    return f"""
    select
      cast(e.party_id as string) as cdi_id,
      cast(e.cmo_ext_party_source_id as string) as cft_id
    from cdiul.ext_id_org e
    where cast(e.party_id as string) in ({cdi_sql_list})
      and upper(cast(e.cmo_ext_source_system as string)) like 'CFT%'
    """


if not cdi_values:
    cft_map_df = pd.DataFrame(columns=['cdi_id', 'cft_id'])
else:
    cft_map_df = _run_impala_fetch(imp, _build_sql_cft(cdi_values), mem_limit='8g')

if cft_map_df is None:
    cft_map_df = pd.DataFrame(columns=['cdi_id', 'cft_id'])
if not cft_map_df.empty:
    cft_map_df['cdi_id'] = cft_map_df['cdi_id'].astype(str)
    cft_map_df['cft_id'] = cft_map_df['cft_id'].astype(str)
    cft_map_df = cft_map_df.drop_duplicates(subset=['cdi_id'], keep='first')

print(f'03_cft_map rows: {len(cft_map_df):,}')

## Секция 4. Операционные метрики
Кратко: считаем `retl_cnt`, `term_cnt`, `amortization` (hybrid 547-like + fallback old).

In [ ]:
if sa_df.empty:
    cmp_df = pd.DataFrame(columns=['n_agr', 'n_cmp_client', 'retl_cnt', 'term_cnt', 'amortization', 'term_calc_source'])
else:
    def _build_sql_cmp_hybrid(use_m_acq_filter=True):
        m_acq_filter_sql = "and coalesce(upper(trim(cast(m.acq_class as string))), 'NA') <> 'SV'" if use_m_acq_filter else ''

        return f"""
        with sa_agr as (
          select distinct
            cast(a.n_agr as string) as n_agr,
            cast(a.n_cmp_client as string) as n_cmp_client
          from ods_alpha.scd1_agreements a
          join ods_alpha.scd1_companies c
            on c.n_cmp = a.n_cmp_client
          where upper(trim(cast(a.acq_class as string))) = 'SA'
            and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
            and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
            and coalesce(a.ods_deleted_flg, '0') <> '1'
            and coalesce(c.ods_deleted_flg, '0') <> '1'
            and c.c_inn is not null
            and exists (
              select 1
              from ods_alpha.scd1_agr_terms t
              where cast(t.n_agr as string) = cast(a.n_agr as string)
                and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
                and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
                and upper(trim(cast(t.cf_ter_type as string))) = 'P'
                and coalesce(t.ods_deleted_flg, '0') <> '1'
            )
        ),

        terms_547 as (
          select distinct
            sa.n_agr,
            sa.n_cmp_client,
            cast(t.c_nmrc as string) as c_nmrc
          from sa_agr sa
          join ods_alpha.scd1_agr_terms t
            on cast(t.n_agr as string) = sa.n_agr
          join ods_alpha.scd1_merchants m
            on cast(m.c_nmrc as string) = cast(t.c_nmrc as string)
          where t.c_nmrc is not null
            and upper(coalesce(trim(cast(m.c_mrc_name as string)), '')) not like 'REZERVNYI TERMINAL%'
            {m_acq_filter_sql}
            and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
            and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{month_start}' as date))
            and coalesce(t.ods_deleted_flg, '0') <> '1'
            and coalesce(m.ods_deleted_flg, '0') <> '1'
        ),
        pos_period as (
          select distinct
            cast(p.c_nmrc as string) as c_nmrc,
            cast(p.c_pos_serial as string) as c_pos_serial,
            cast(p.c_nter as string) as c_nter
          from ods_alpha.scd1_pos_terminals p
          where p.c_pos_serial is not null
            and cast(p.d_ter_install as date) is not null
            and cast(p.d_ter_install as date) < date_add(cast('{month_end}' as date), 1)
            and (p.d_ter_close is null or cast(p.d_ter_close as date) >= cast('{month_start}' as date))
            and coalesce(p.ods_deleted_flg, '0') <> '1'
        ),
        term_active_547 as (
          select
            t.n_agr,
            t.n_cmp_client,
            t.c_nmrc,
            p.c_pos_serial,
            p.c_nter
          from terms_547 t
          left join pos_period p
            on p.c_nmrc = t.c_nmrc
        ),
        retl_547 as (
          select n_agr, count(distinct c_nmrc) as retl_cnt_547
          from term_active_547
          group by n_agr
        ),
        term_547 as (
          select n_agr, count(distinct c_pos_serial) as term_cnt_547
          from term_active_547
          group by n_agr
        ),
        amort_547 as (
          select x.n_agr, sum(coalesce(cast(am.amortization_monthly as double), 0.0)) as amortization_547
          from (
            select distinct n_agr, c_nter
            from term_active_547
            where c_nter is not null
          ) x
          left join sandbox_ai.shestopalov_terminal_amortization_oneoff am
            on cast(am.c_nter as string) = x.c_nter
          group by x.n_agr
        ),

        fallback_agrs as (
          select sa.n_agr, sa.n_cmp_client
          from sa_agr sa
          left join retl_547 r547 on r547.n_agr = sa.n_agr
          left join term_547 t547 on t547.n_agr = sa.n_agr
          where r547.n_agr is null and t547.n_agr is null
        ),
        old_nmrc as (
          select fa.n_agr, fa.n_cmp_client, cast(mm.c_nmrc as string) as c_nmrc
          from fallback_agrs fa
          join ods_alpha.scd1_merchants mm
            on cast(mm.n_cmp as string) = fa.n_cmp_client
          where mm.c_nmrc is not null
            and coalesce(mm.ods_deleted_flg, '0') <> '1'
          group by fa.n_agr, fa.n_cmp_client, cast(mm.c_nmrc as string)
        ),
        old_term_active as (
          select
            o.n_agr,
            o.n_cmp_client,
            o.c_nmrc,
            cast(t.c_nter as string) as c_nter
          from old_nmrc o
          join ods_alpha.scd1_pos_terminals t
            on cast(t.c_nmrc as string) = o.c_nmrc
          where t.c_nter is not null
            and coalesce(t.ods_deleted_flg, '0') <> '1'
            and cast(t.d_ter_install as date) is not null
            and cast(t.d_ter_install as date) <= cast('{month_end}' as date)
            and coalesce(cast(t.d_ter_close as date), cast('2999-12-31' as date)) >= cast('{month_start}' as date)
          group by o.n_agr, o.n_cmp_client, o.c_nmrc, cast(t.c_nter as string)
        ),
        old_retl as (
          select n_agr, count(distinct c_nmrc) as retl_cnt_old
          from old_term_active
          group by n_agr
        ),
        old_term as (
          select n_agr, count(distinct c_nter) as term_cnt_old
          from old_term_active
          group by n_agr
        ),
        old_amort as (
          select x.n_agr, sum(coalesce(cast(am.amortization_monthly as double), 0.0)) as amortization_old
          from (
            select distinct n_agr, c_nter
            from old_term_active
            where c_nter is not null
          ) x
          left join sandbox_ai.shestopalov_terminal_amortization_oneoff am
            on cast(am.c_nter as string) = x.c_nter
          group by x.n_agr
        )

        select
          sa.n_agr,
          sa.n_cmp_client,
          coalesce(r547.retl_cnt_547, rold.retl_cnt_old) as retl_cnt,
          coalesce(t547.term_cnt_547, told.term_cnt_old) as term_cnt,
          coalesce(a547.amortization_547, aold.amortization_old, 0.0) as amortization,
          case
            when r547.n_agr is not null or t547.n_agr is not null then '547_like'
            when rold.n_agr is not null or told.n_agr is not null then 'fallback_old'
            else 'no_data'
          end as term_calc_source
        from sa_agr sa
        left join retl_547 r547 on r547.n_agr = sa.n_agr
        left join term_547 t547 on t547.n_agr = sa.n_agr
        left join amort_547 a547 on a547.n_agr = sa.n_agr
        left join old_retl rold on rold.n_agr = sa.n_agr
        left join old_term told on told.n_agr = sa.n_agr
        left join old_amort aold on aold.n_agr = sa.n_agr
        """

    try:
        cmp_df = _run_impala_fetch(imp, _build_sql_cmp_hybrid(use_m_acq_filter=True), mem_limit='16g')
    except Exception:
        cmp_df = _run_impala_fetch(imp, _build_sql_cmp_hybrid(use_m_acq_filter=False), mem_limit='16g')

if cmp_df is None:
    cmp_df = pd.DataFrame(columns=['n_agr', 'n_cmp_client', 'retl_cnt', 'term_cnt', 'amortization', 'term_calc_source'])
if not cmp_df.empty:
    cmp_df['n_agr'] = cmp_df['n_agr'].astype(str)
    cmp_df['n_cmp_client'] = cmp_df['n_cmp_client'].astype(str)
    cmp_df['retl_cnt'] = pd.to_numeric(cmp_df['retl_cnt'], errors='coerce')
    cmp_df['term_cnt'] = pd.to_numeric(cmp_df['term_cnt'], errors='coerce')
    cmp_df['amortization'] = pd.to_numeric(cmp_df['amortization'], errors='coerce')

print(f'04_operational_metrics rows: {len(cmp_df):,}')

## Секция 5. Транзакционные метрики
Кратко: считаем `trx_cnt`, `trx_sum`, `commission_from_ops`, `int_component`, активные терминалы.

In [ ]:
if sa_df.empty:
    trx_df = pd.DataFrame(columns=['n_agr', 'inn', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component', 'n_cmp_client', 'active_term_cnt', 'active_terms'])
else:
    sql_trx = f"""
    with sa_agr as (
      select distinct
        cast(a.n_agr as string) as n_agr,
        cast(a.n_cmp_client as string) as n_cmp_client,
        regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn
      from ods_alpha.scd1_agreements a
      join ods_alpha.scd1_companies c
        on c.n_cmp = a.n_cmp_client
      where upper(trim(cast(a.acq_class as string))) = 'SA'
        and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
        and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
        and coalesce(a.ods_deleted_flg, '0') <> '1'
        and coalesce(c.ods_deleted_flg, '0') <> '1'
        and c.c_inn is not null
        and exists (
          select 1
          from ods_alpha.scd1_agr_terms t
          where cast(t.n_agr as string) = cast(a.n_agr as string)
            and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
            and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
            and upper(trim(cast(t.cf_ter_type as string))) = 'P'
            and coalesce(t.ods_deleted_flg, '0') <> '1'
        )
    ),
    fiid_rshb as (
      select distinct cast(fa.c_fiid as string) as c_fiid
      from ods_alpha.scd1_base24_fiids fa
      where coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
    ),
    trx_base_raw as (
      select cast(t.n_trx as string) as n_trx, cast(t.c_nter as string) as c_nter, cast(t.n_amt_src as double) as n_amt_src
      from ods_alpha.scd1_trx t
      join fiid_rshb fr
        on fr.c_fiid = cast(t.c_fiid_acq as string)
      where cast(t.d_trx_orig as timestamp) >= cast('{month_start}' as timestamp)
        and cast(t.d_trx_orig as timestamp) < cast(date_add(cast('{month_end}' as date), 1) as timestamp)
        and t.c_nter is not null
        and coalesce(t.ods_deleted_flg, '0') <> '1'
        and t.c_trx_class = 'SA'
        and t.c_trx_type = 'S01'
        and coalesce(t.cf_trx_stat, '') <> 'R'
    ),
    trx_base as (
      select n_trx, max(c_nter) as c_nter, max(n_amt_src) as n_amt_src
      from trx_base_raw
      group by n_trx
    ),
    ta_raw as (
      select cast(a.n_trx as string) as n_trx, cast(a.n_agr as string) as n_agr, coalesce(cast(a.n_amt_tax as double), 0.0) as n_amt_tax
      from ods_alpha.scd1_trx_acq a
      join trx_base tb on tb.n_trx = cast(a.n_trx as string)
      join sa_agr ss on ss.n_agr = cast(a.n_agr as string)
    ),
    ta as (
      select n_trx, n_agr, max(n_amt_tax) as n_amt_tax
      from ta_raw
      group by n_trx, n_agr
    ),
    trx_keys as (
      select distinct n_trx
      from ta
    ),
    trx_int_agg as (
      select cast(i.n_trx as string) as n_trx, sum(coalesce(cast(i.n_amt_fee as double), 0.0)) as n_amt_fee
      from ods_alpha.scd1_trx_int i
      join trx_keys k on k.n_trx = cast(i.n_trx as string)
      group by cast(i.n_trx as string)
    ),
    tj as (
      select ta.n_agr, sa.n_cmp_client, sa.inn, ta.n_trx, tb.c_nter, tb.n_amt_src, ta.n_amt_tax
      from ta
      join trx_base tb on tb.n_trx = ta.n_trx
      left join sa_agr sa on sa.n_agr = ta.n_agr
    ),
    trx_agg as (
      select
        tj.n_agr,
        count(distinct tj.n_trx) as trx_cnt,
        sum(tj.n_amt_src) as trx_sum,
        sum(tj.n_amt_tax) as commission_from_ops,
        sum(coalesce(i.n_amt_fee, 0.0)) as int_component
      from tj
      left join trx_int_agg i on i.n_trx = tj.n_trx
      group by tj.n_agr
    ),
    active_term_agg_ncmp as (
      select
        tj.n_cmp_client,
        count(distinct case when coalesce(tj.n_amt_src, 0.0) > 1 then tj.c_nter else null end) as active_term_cnt
      from tj
      where tj.n_cmp_client is not null
      group by tj.n_cmp_client
    ),
    active_term_agg_agr as (
      select
        tj.n_agr,
        count(distinct case when coalesce(tj.n_amt_src, 0.0) > 1 then tj.c_nter else null end) as active_terms
      from tj
      where tj.n_agr is not null
      group by tj.n_agr
    )
    select
      t.n_agr,
      sa.inn,
      t.trx_cnt,
      t.trx_sum,
      t.commission_from_ops,
      t.int_component,
      sa.n_cmp_client,
      a.active_term_cnt,
      aa.active_terms
    from trx_agg t
    left join sa_agr sa on sa.n_agr = t.n_agr
    left join active_term_agg_ncmp a on a.n_cmp_client = sa.n_cmp_client
    left join active_term_agg_agr aa on aa.n_agr = t.n_agr
    """

    trx_df = _run_impala_fetch(imp, sql_trx, mem_limit='16g')

if trx_df is None:
    trx_df = pd.DataFrame(columns=['n_agr', 'inn', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component', 'n_cmp_client', 'active_term_cnt', 'active_terms'])
if not trx_df.empty:
    trx_df['n_agr'] = trx_df['n_agr'].astype(str)
    trx_df['n_cmp_client'] = trx_df['n_cmp_client'].astype(str)
    trx_df['inn'] = trx_df['inn'].map(normalize_inn)
    trx_df['active_term_cnt'] = pd.to_numeric(trx_df['active_term_cnt'], errors='coerce')
    trx_df['active_terms'] = pd.to_numeric(trx_df['active_terms'], errors='coerce')

active_term_df = (
    trx_df[['n_cmp_client', 'active_term_cnt']]
    .dropna(subset=['n_cmp_client'])
    .drop_duplicates(subset=['n_cmp_client'])
    if len(trx_df)
    else pd.DataFrame(columns=['n_cmp_client', 'active_term_cnt'])
)

print(f'05_transaction_metrics rows: {len(trx_df):,}')

## Секция 6. R2 legacy attrs
Кратко: по `cft_id` подтягиваем R2-профиль, legacy-тариф и fallback-поля филиала/ВСП.

In [ ]:
cft_values = sorted([
    x for x in cft_map_df.get('cft_id', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x
])

if not cft_values:
    r2_df = pd.DataFrame(columns=['r2_id', 'cft_id', 'c_tariff_plan', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf_raw', 'filial_rf', 'tariff_name_legacy', 'commission_monthly_legacy'])
    r2_profile_cft_df = pd.DataFrame(columns=['cft_id', 'filial_rf_cft_fb', 'vsp_name_cft_fb', 'vsp_code_cft_fb'])
    r2_profile_r2_df = pd.DataFrame(columns=['r2_id', 'filial_rf_r2_fb', 'vsp_name_r2_fb', 'vsp_code_r2_fb'])
else:
    def _split_cft_scope(cft_scope):
        numeric_scope = []
        text_scope = []
        for raw_v in (cft_scope or []):
            v = str(raw_v).strip()
            if not v:
                continue
            if v.isdigit():
                numeric_scope.append(v)
            else:
                text_scope.append(v)
        return sorted(set(numeric_scope)), sorted(set(text_scope))

    def _build_sql_r2_scoped(cft_scope):
        cft_numeric_scope, cft_text_scope = _split_cft_scope(cft_scope)

        scope_predicates = []
        if cft_numeric_scope:
            scope_predicates.append(f"m.c_cl_org in ({', '.join(cft_numeric_scope)})")
        if cft_text_scope:
            cft_text_sql_list = _in_sql_list(cft_text_scope)
            scope_predicates.append(f"cast(m.c_cl_org as string) in ({cft_text_sql_list})")

        scope_predicate_sql = ' or '.join([f"({p})" for p in scope_predicates]) if scope_predicates else '1 = 0'

        return f"""
        with merchant_ranked as (
          select
            m.id as r2_id_raw,
            m.c_cl_org as cft_id_raw,
            m.c_depart as c_depart_raw,
            m.c_tariff_plan as c_tariff_plan_raw,
            row_number() over (
              partition by m.c_cl_org
              order by
                case when m.c_tariff_plan is not null then 0 else 1 end,
                case when m.c_depart is not null then 0 else 1 end,
                m.id desc
            ) as rn
          from ods.scd1_z_r2_ip_merchants m
          where ({scope_predicate_sql})
            and m.c_cl_org is not null
            and coalesce(m.ods_deleted_flg, '0') <> '1'
        ),
        r2_raw as (
          select
            r2_id_raw,
            cft_id_raw,
            c_depart_raw,
            c_tariff_plan_raw
          from merchant_ranked
          where rn = 1
        ),
        r2_enriched as (
          select
            r2.r2_id_raw as r2_id,
            r2.cft_id_raw as cft_id,
            r2.c_tariff_plan_raw as c_tariff_plan,
            corp.c_register_gos_reg_num_rec as ogrn,
            dep.c_name as vsp_name,
            dep.c_num as vsp_code,
            br.c_shortlabel as filial_rf_raw,
            tp.c_name as tariff_name_legacy,
            cast(null as decimal(18,2)) as commission_monthly_legacy
          from r2_raw r2
          left join ods.scd1_z_cl_corp corp on corp.id = r2.cft_id_raw
          left join ods.scd1_z_depart dep on dep.id = r2.c_depart_raw
          left join ods.scd1_z_branch br on br.id = dep.c_filial
          left join ods.scd1_z_r2_tariff_plan tp on tp.id = r2.c_tariff_plan_raw
        )
        select
          r2_id,
          cft_id,
          c_tariff_plan,
          ogrn,
          vsp_name,
          vsp_code,
          filial_rf_raw,
          tariff_name_legacy,
          commission_monthly_legacy
        from r2_enriched
        """

    def _build_sql_r2_full(cft_scope):
        cft_sql_list = _in_sql_list(cft_scope)
        return f"""
        with merchant_ranked as (
          select
            m.id as r2_id_raw,
            m.c_cl_org as cft_id_raw,
            m.c_depart as c_depart_raw,
            m.c_tariff_plan as c_tariff_plan_raw,
            row_number() over (
              partition by cast(m.c_cl_org as string)
              order by
                case when m.c_tariff_plan is not null then 0 else 1 end,
                case when m.c_depart is not null then 0 else 1 end,
                m.id desc
            ) as rn
          from ods.scd1_z_r2_ip_merchants m
          where cast(m.c_cl_org as string) in ({cft_sql_list})
            and m.c_cl_org is not null
            and coalesce(m.ods_deleted_flg, '0') <> '1'
        ),
        r2_raw as (
          select
            r2_id_raw,
            cft_id_raw,
            c_depart_raw,
            c_tariff_plan_raw
          from merchant_ranked
          where rn = 1
        ),
        r2_enriched as (
          select
            r2.r2_id_raw as r2_id,
            r2.cft_id_raw as cft_id,
            r2.c_tariff_plan_raw as c_tariff_plan,
            corp.c_register_gos_reg_num_rec as ogrn,
            dep.c_name as vsp_name,
            dep.c_num as vsp_code,
            br.c_shortlabel as filial_rf_raw,
            tp.c_name as tariff_name_legacy,
            cast(null as decimal(18,2)) as commission_monthly_legacy
          from r2_raw r2
          left join ods.scd1_z_cl_corp corp on corp.id = r2.cft_id_raw
          left join ods.scd1_z_depart dep on dep.id = r2.c_depart_raw
          left join ods.scd1_z_branch br on br.id = dep.c_filial
          left join ods.scd1_z_r2_tariff_plan tp on tp.id = r2.c_tariff_plan_raw
        )
        select
          r2_id,
          cft_id,
          c_tariff_plan,
          ogrn,
          vsp_name,
          vsp_code,
          filial_rf_raw,
          tariff_name_legacy,
          commission_monthly_legacy
        from r2_enriched
        """

    try:
        r2_df = _run_impala_fetch(imp, _build_sql_r2_scoped(cft_values), mem_limit='8g')
    except Exception:
        r2_df = _run_impala_fetch(imp, _build_sql_r2_full(cft_values), mem_limit='8g')

    if r2_df is None:
        r2_df = pd.DataFrame(columns=['r2_id', 'cft_id', 'c_tariff_plan', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf_raw', 'filial_rf', 'tariff_name_legacy', 'commission_monthly_legacy'])

    if not r2_df.empty:
        for c in ['r2_id', 'cft_id', 'c_tariff_plan', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf_raw', 'tariff_name_legacy']:
            r2_df[c] = r2_df[c].astype(str)

        r2_df['filial_rf_raw'] = r2_df['filial_rf_raw'].apply(normalize_text_value)
        r2_df['filial_rf'] = r2_df['filial_rf_raw'].apply(normalize_filial_rf)

        if 'commission_monthly_legacy' not in r2_df.columns:
            r2_df['commission_monthly_legacy'] = None
        r2_df['commission_monthly_legacy'] = pd.to_numeric(r2_df.get('commission_monthly_legacy'), errors='coerce')

        r2_profile_cft_df = (
            r2_df.groupby('cft_id', as_index=False)
            .agg(
                filial_rf_cft_fb=('filial_rf', first_non_empty_value),
                vsp_name_cft_fb=('vsp_name', first_non_empty_value),
                vsp_code_cft_fb=('vsp_code', first_non_empty_value),
            )
        )
        r2_profile_r2_df = (
            r2_df.groupby('r2_id', as_index=False)
            .agg(
                filial_rf_r2_fb=('filial_rf', first_non_empty_value),
                vsp_name_r2_fb=('vsp_name', first_non_empty_value),
                vsp_code_r2_fb=('vsp_code', first_non_empty_value),
            )
        )

        r2_df = r2_df.drop_duplicates(subset=['cft_id'], keep='first')
    else:
        r2_profile_cft_df = pd.DataFrame(columns=['cft_id', 'filial_rf_cft_fb', 'vsp_name_cft_fb', 'vsp_code_cft_fb'])
        r2_profile_r2_df = pd.DataFrame(columns=['r2_id', 'filial_rf_r2_fb', 'vsp_name_r2_fb', 'vsp_code_r2_fb'])

print(f'06_r2_legacy_attrs rows: {len(r2_df):,}')

## Секция 7. Сборка базовой витрины
Кратко: объединяем SA/CDI/CFT/R2/операционные/транзакционные данные в `base_df`.

In [ ]:
def _is_blank_series(s):
    return s.isna() | s.astype(str).str.strip().isin(['', 'nan', 'None'])


base_df = sa_df.copy()

if not cdi_map_df.empty and not base_df.empty:
    cdi_merge_cols = ['inn', 'cdi_id', 'ssp_ocrm']
    for extra_col in ['ssp_ocrm_raw', 'x_area_resp_norm']:
        if extra_col in cdi_map_df.columns:
            cdi_merge_cols.append(extra_col)
    base_df = base_df.merge(cdi_map_df[cdi_merge_cols], on='inn', how='left')
else:
    base_df['cdi_id'] = None
    base_df['ssp_ocrm'] = None
    base_df['ssp_ocrm_raw'] = None
    base_df['x_area_resp_norm'] = None

if 'ssp_ocrm_raw' in base_df.columns:
    ssp_from_raw = base_df['ssp_ocrm_raw'].apply(normalize_ssp_ocrm_core)
    base_df['ssp_ocrm'] = ssp_from_raw.where(ssp_from_raw.notna(), base_df['ssp_ocrm'])
base_df['ssp_ocrm'] = base_df['ssp_ocrm'].apply(normalize_ssp_ocrm_core)

if not cft_map_df.empty and not base_df.empty:
    base_df = base_df.merge(cft_map_df[['cdi_id', 'cft_id']], on='cdi_id', how='left')
else:
    base_df['cft_id'] = None

if not r2_df.empty and not base_df.empty:
    base_df = base_df.merge(r2_df, on='cft_id', how='left')
else:
    for col in ['r2_id', 'c_tariff_plan', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf_raw', 'filial_rf', 'tariff_name_legacy', 'commission_monthly_legacy']:
        base_df[col] = None

if not base_df.empty and 'cft_id' in base_df.columns and not r2_profile_cft_df.empty:
    base_df = base_df.merge(r2_profile_cft_df, on='cft_id', how='left')
else:
    for col in ['filial_rf_cft_fb', 'vsp_name_cft_fb', 'vsp_code_cft_fb']:
        base_df[col] = None

if not base_df.empty and 'r2_id' in base_df.columns and not r2_profile_r2_df.empty:
    base_df = base_df.merge(r2_profile_r2_df, on='r2_id', how='left')
else:
    for col in ['filial_rf_r2_fb', 'vsp_name_r2_fb', 'vsp_code_r2_fb']:
        base_df[col] = None

for col, cft_fb, r2_fb in [
    ('filial_rf', 'filial_rf_cft_fb', 'filial_rf_r2_fb'),
    ('vsp_name', 'vsp_name_cft_fb', 'vsp_name_r2_fb'),
    ('vsp_code', 'vsp_code_cft_fb', 'vsp_code_r2_fb'),
]:
    missing_mask = _is_blank_series(base_df[col])
    base_df.loc[missing_mask, col] = base_df.loc[missing_mask, cft_fb]

    missing_mask = _is_blank_series(base_df[col])
    base_df.loc[missing_mask, col] = base_df.loc[missing_mask, r2_fb]

base_df['filial_rf_raw'] = base_df['filial_rf_raw'].apply(normalize_text_value)
base_df['filial_rf'] = base_df['filial_rf'].apply(normalize_filial_rf)

for fb_col in ['filial_rf_cft_fb', 'vsp_name_cft_fb', 'vsp_code_cft_fb', 'filial_rf_r2_fb', 'vsp_name_r2_fb', 'vsp_code_r2_fb']:
    if fb_col in base_df.columns:
        base_df = base_df.drop(columns=[fb_col])

if not cmp_df.empty and not base_df.empty:
    base_df = base_df.merge(cmp_df[['n_agr', 'retl_cnt', 'term_cnt', 'amortization']], on='n_agr', how='left')
else:
    for col in ['retl_cnt', 'term_cnt', 'amortization']:
        base_df[col] = None

if not active_term_df.empty and not base_df.empty:
    base_df = base_df.merge(active_term_df[['n_cmp_client', 'active_term_cnt']], on='n_cmp_client', how='left')
else:
    base_df['active_term_cnt'] = None

if not trx_df.empty and not base_df.empty:
    base_df = base_df.merge(
        trx_df[['n_agr', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component', 'active_terms']],
        on='n_agr',
        how='left'
    )
else:
    for col in ['trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component', 'active_terms']:
        base_df[col] = None

print(f'07_base_merge rows: {len(base_df):,}')

## Секция 8. agr_id fallback
Кратко: если `agr_id` пустой, подставляем `r2_id`.

In [ ]:
if 'agr_id' not in base_df.columns:
    base_df['agr_id'] = None
if 'r2_id' not in base_df.columns:
    base_df['r2_id'] = None

agr_before_mask = base_df['agr_id'].notna() & (base_df['agr_id'].astype(str).str.strip() != '')
base_df['agr_id_source'] = 'sa'
fallback_mask = (~agr_before_mask) & base_df['r2_id'].notna() & (base_df['r2_id'].astype(str).str.strip() != '')
base_df.loc[fallback_mask, 'agr_id'] = base_df.loc[fallback_mask, 'r2_id']
base_df.loc[fallback_mask, 'agr_id_source'] = 'r2_fallback'

print(f'08_agr_fallback applied rows: {int(fallback_mask.sum()):,}')

## Секция 8b. Scoped flat map тарифа
Кратко: строим map `c_tariff_plan -> commission_monthly_fix` только по планам текущего месяца.

In [ ]:
agr_id_scope = (
    base_df.get('agr_id', pd.Series(dtype=object))
    .map(normalize_agr_q1)
    .dropna()
    .astype(str)
    .str.strip()
)
agr_id_scope = sorted([x for x in agr_id_scope.unique().tolist() if x])

plan_scope_08b = sorted([
    x for x in base_df.get('c_tariff_plan', pd.Series(dtype=object)).dropna().astype(str).str.strip().unique().tolist()
    if x and x.lower() not in {'nan', 'none', 'null'}
])


def _build_sql_tariff_fix_map_scoped(plan_scope_values):
    plan_in = _in_sql_list(plan_scope_values)
    return f"""
    with tt_pairs as (
      select distinct
        tt.c_tariff_plan as c_tariff_plan_raw,
        tt.c_tariff as c_tariff_raw
      from ods.scd1_z_r2_tariff_tune tt
      where cast(tt.c_tariff_plan as string) in ({plan_in})
    ),
    tf_agg as (
      select
        tf.id as c_tariff_raw,
        max(cast(tf.c_summa as decimal(18,2))) as commission_monthly_fix
      from ods.scd1_z_r2_tariff_fix tf
      group by tf.id
    )
    select
      cast(p.c_tariff_plan_raw as string) as c_tariff_plan,
      max(a.commission_monthly_fix) as commission_monthly_fix
    from tt_pairs p
    left join tf_agg a
      on p.c_tariff_raw = a.c_tariff_raw
    group by p.c_tariff_plan_raw
    """


if disable_flatmap_runtime_cache:
    tariff_fix_map_flat_df = pd.DataFrame(columns=['c_tariff_plan', 'commission_monthly_fix'])
    if '_tariff_fix_map_runtime_cache_df' in globals():
        globals().pop('_tariff_fix_map_runtime_cache_df', None)

if not plan_scope_08b:
    tariff_fix_map_df = pd.DataFrame(columns=['c_tariff_plan', 'commission_monthly_fix'])
    map_source = 'empty_plan_scope'
elif use_fast_flat_08b_map and (not disable_flatmap_runtime_cache) and tariff_fix_map_flat_df is not None and len(tariff_fix_map_flat_df):
    tariff_fix_map_df = tariff_fix_map_flat_df.copy()
    map_source = 'preloaded_flat_map_scoped'
elif use_fast_flat_08b_map and (not disable_flatmap_runtime_cache) and '_tariff_fix_map_runtime_cache_df' in globals() and len(globals()['_tariff_fix_map_runtime_cache_df']):
    tariff_fix_map_df = globals()['_tariff_fix_map_runtime_cache_df'].copy()
    map_source = 'runtime_cached_flat_map_scoped'
elif use_fast_flat_08b_map:
    tariff_fix_map_df = _run_impala_fetch(
        imp,
        _build_sql_tariff_fix_map_scoped(plan_scope_08b),
        mem_limit='8g',
    )
    if not disable_flatmap_runtime_cache:
        globals()['_tariff_fix_map_runtime_cache_df'] = tariff_fix_map_df.copy() if tariff_fix_map_df is not None else pd.DataFrame(columns=['c_tariff_plan', 'commission_monthly_fix'])
    map_source = 'rebuilt_scoped_map_nocache' if disable_flatmap_runtime_cache else 'rebuilt_scoped_map'
else:
    tariff_fix_map_df = _run_impala_fetch(
        imp,
        _build_sql_tariff_fix_map_scoped(plan_scope_08b),
        mem_limit='8g',
    )
    map_source = 'legacy_scoped_map'

if tariff_fix_map_df is None:
    tariff_fix_map_df = pd.DataFrame(columns=['c_tariff_plan', 'commission_monthly_fix'])

if not tariff_fix_map_df.empty:
    tariff_fix_map_df['c_tariff_plan'] = tariff_fix_map_df['c_tariff_plan'].astype(str).str.strip()
    tariff_fix_map_df['commission_monthly_fix'] = pd.to_numeric(
        tariff_fix_map_df.get('commission_monthly_fix'), errors='coerce'
    )

    plan_scope_set = set(plan_scope_08b)
    tariff_fix_map_df = tariff_fix_map_df[
        tariff_fix_map_df['c_tariff_plan'].isin(plan_scope_set)
    ].copy()

    tariff_fix_map_df = (
        tariff_fix_map_df
        .dropna(subset=['c_tariff_plan'])
        .groupby('c_tariff_plan', as_index=False)['commission_monthly_fix']
        .max()
        .sort_values('c_tariff_plan')
        .reset_index(drop=True)
    )
    plan_scope = sorted([
        x for x in tariff_fix_map_df['c_tariff_plan'].dropna().astype(str).str.strip().unique().tolist()
        if x and x.lower() not in {'nan', 'none', 'null'}
    ])
else:
    plan_scope = []

print(
    f"08b scoped map: source={map_source}, agr_scope={len(agr_id_scope):,}, "
    f"plan_scope_input={len(plan_scope_08b):,}, plan_scope={len(plan_scope):,}, rows={len(tariff_fix_map_df):,}"
)

## Секция 9. Актуальный тариф по agr_id
Кратко: определяем актуальный `c_tariff_plan` и `tariff_name` по `agr_id`, затем строим карту `commission_monthly_actual`.

In [ ]:
base_df['agr_id_key'] = base_df['agr_id'].map(normalize_agr_q1)
base_agr_keys_df = (
    base_df[['agr_id_key']]
    .dropna()
    .drop_duplicates()
    .rename(columns={'agr_id_key': 'agr_id'})
)

base_agr_scope = []
actual_plan_scope = []
actual_tariff_fetch_mode = 'empty_scope'
commission_map_source = 'empty_scope'


def _build_sql_actual_tariff_by_agr(agr_scope):
    agr_id_in = _in_sql_list(agr_scope)
    return f"""
    with agr_actual as (
      select
        cast(a.abs_agr_id as string) as agr_id,
        cast(a.n_agr as string) as n_agr_actual,
        cast(a.c_agr_number as string) as contract_number_acq,
        cast(a.d_valid_from as date) as d_valid_from_actual,
        cast(a.d_valid_to as date) as d_valid_to_actual,
        row_number() over (
          partition by cast(a.abs_agr_id as string)
          order by cast(a.d_valid_from as date) desc, cast(a.n_agr as string) desc
        ) as rn
      from ods_alpha.scd1_agreements a
      where cast(a.abs_agr_id as string) in ({agr_id_in})
        and upper(trim(cast(a.acq_class as string))) = 'SA'
        and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
        and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_end}' as date))
        and coalesce(a.ods_deleted_flg, '0') <> '1'
    ),
    merchant_one as (
      select
        cast(m.id as string) as agr_id,
        cast(m.c_tariff_plan as string) as c_tariff_plan,
        row_number() over (
          partition by cast(m.id as string)
          order by cast(m.c_tariff_plan as string) desc
        ) as rn
      from ods.scd1_z_r2_ip_merchants m
      where cast(m.id as string) in ({agr_id_in})
        and coalesce(m.ods_deleted_flg, '0') <> '1'
    )
    select
      m.agr_id,
      aa.n_agr_actual,
      aa.contract_number_acq,
      aa.d_valid_from_actual,
      aa.d_valid_to_actual,
      m.c_tariff_plan,
      cast(tp.c_name as string) as tariff_name_actual
    from merchant_one m
    left join agr_actual aa
      on aa.agr_id = m.agr_id
     and aa.rn = 1
    left join ods.scd1_z_r2_tariff_plan tp
      on cast(tp.id as string) = m.c_tariff_plan
    where m.rn = 1
    """


if len(base_agr_keys_df):
    base_agr_scope = sorted(
        base_agr_keys_df['agr_id']
        .dropna()
        .astype(str)
        .str.strip()
        .loc[lambda s: s != '']
        .unique()
        .tolist()
    )

    if len(base_agr_scope) > int(section09_chunk_threshold):
        scope_chunks = _split_scope(base_agr_scope, section09_chunk_size)
        actual_tariff_fetch_mode = f"chunked_{len(scope_chunks)}"
        actual_tariff_parts = []
        for agr_chunk in scope_chunks:
            chunk_df = _run_impala_fetch(
                imp,
                _build_sql_actual_tariff_by_agr(agr_chunk),
                mem_limit='12g',
            )
            if chunk_df is not None and len(chunk_df):
                actual_tariff_parts.append(chunk_df)

        if actual_tariff_parts:
            actual_tariff_df = pd.concat(actual_tariff_parts, ignore_index=True)
        else:
            actual_tariff_df = pd.DataFrame(columns=[
                'agr_id', 'n_agr_actual', 'contract_number_acq', 'd_valid_from_actual', 'd_valid_to_actual',
                'c_tariff_plan', 'tariff_name_actual'
            ])
    else:
        actual_tariff_fetch_mode = 'single_shot'
        actual_tariff_df = _run_impala_fetch(
            imp,
            _build_sql_actual_tariff_by_agr(base_agr_scope),
            mem_limit='12g'
        )
else:
    actual_tariff_df = pd.DataFrame(columns=[
        'agr_id', 'n_agr_actual', 'contract_number_acq', 'd_valid_from_actual', 'd_valid_to_actual',
        'c_tariff_plan', 'tariff_name_actual'
    ])

if actual_tariff_df is None:
    actual_tariff_df = pd.DataFrame(columns=[
        'agr_id', 'n_agr_actual', 'contract_number_acq', 'd_valid_from_actual', 'd_valid_to_actual',
        'c_tariff_plan', 'tariff_name_actual'
    ])

if not actual_tariff_df.empty:
    for c in ['agr_id', 'n_agr_actual', 'contract_number_acq', 'c_tariff_plan', 'tariff_name_actual']:
        actual_tariff_df[c] = actual_tariff_df[c].astype(str).str.strip()
    actual_tariff_df = actual_tariff_df.merge(base_agr_keys_df, on='agr_id', how='inner')

actual_tariff_df = actual_tariff_df.rename(columns={'agr_id': 'agr_id_key'})

if 'c_tariff_plan' not in actual_tariff_df.columns:
    actual_tariff_df['c_tariff_plan'] = None

actual_comm_map_df = pd.DataFrame(columns=['c_tariff_plan', 'commission_monthly_fix'])
if not actual_tariff_df.empty:
    actual_plan_scope = sorted([
        x for x in actual_tariff_df['c_tariff_plan'].dropna().astype(str).str.strip().unique().tolist()
        if x and x.lower() not in {'nan', 'none', 'null'}
    ])

    if actual_plan_scope:
        if tariff_fix_map_df is not None and len(tariff_fix_map_df):
            commission_map_source = '08b_cached_scope_filter'
            actual_comm_map_df = tariff_fix_map_df[
                tariff_fix_map_df['c_tariff_plan'].astype(str).str.strip().isin(set(actual_plan_scope))
            ][['c_tariff_plan', 'commission_monthly_fix']].copy()
        else:
            commission_map_source = '09_sql_scope'
            plan_in = _in_sql_list(actual_plan_scope)
            sql_actual_commission_by_plan = f"""
            with tt_pairs as (
              select distinct
                tt.c_tariff_plan as c_tariff_plan_raw,
                tt.c_tariff as c_tariff_raw
              from ods.scd1_z_r2_tariff_tune tt
              where cast(tt.c_tariff_plan as string) in ({plan_in})
            ),
            tf_agg as (
              select
                tf.id as c_tariff_raw,
                max(cast(tf.c_summa as decimal(18,2))) as commission_monthly_fix
              from ods.scd1_z_r2_tariff_fix tf
              group by tf.id
            )
            select
              cast(p.c_tariff_plan_raw as string) as c_tariff_plan,
              max(a.commission_monthly_fix) as commission_monthly_fix
            from tt_pairs p
            left join tf_agg a
              on p.c_tariff_raw = a.c_tariff_raw
            group by p.c_tariff_plan_raw
            """
            actual_comm_map_df = _run_impala_fetch(
                imp,
                sql_actual_commission_by_plan,
                mem_limit='8g',
            )

if actual_comm_map_df is None:
    actual_comm_map_df = pd.DataFrame(columns=['c_tariff_plan', 'commission_monthly_fix'])

if not actual_comm_map_df.empty:
    actual_comm_map_df['c_tariff_plan'] = actual_comm_map_df['c_tariff_plan'].astype(str).str.strip()
    actual_comm_map_df['commission_monthly_fix'] = pd.to_numeric(
        actual_comm_map_df.get('commission_monthly_fix'), errors='coerce'
    )
    actual_comm_map_df = (
        actual_comm_map_df
        .dropna(subset=['c_tariff_plan'])
        .groupby('c_tariff_plan', as_index=False)['commission_monthly_fix']
        .max()
    )
    actual_tariff_df = actual_tariff_df.merge(actual_comm_map_df, on='c_tariff_plan', how='left')
else:
    actual_tariff_df['commission_monthly_fix'] = None

actual_tariff_df['commission_monthly_actual'] = pd.to_numeric(
    actual_tariff_df.get('commission_monthly_fix'), errors='coerce'
)

if 'commission_monthly_fix' in actual_tariff_df.columns:
    actual_tariff_df = actual_tariff_df.drop(columns=['commission_monthly_fix'])

print(
    f"09_actual_tariff_by_agr: mode={actual_tariff_fetch_mode}, agr_scope={len(base_agr_scope):,}, "
    f"actual_rows={len(actual_tariff_df):,}, actual_plan_scope={len(actual_plan_scope):,}, "
    f"commission_map_source={commission_map_source}"
)

## Секция 10. Формулы и final_df
Кратко: применяем каскады тарифа/комиссии и формируем итоговый `final_df`.

In [ ]:
required_tariff_cols = [
    'agr_id_key', 'n_agr_actual', 'contract_number_acq', 'd_valid_from_actual',
    'd_valid_to_actual', 'c_tariff_plan', 'tariff_name_actual', 'commission_monthly_actual'
]
for col in required_tariff_cols:
    if col not in actual_tariff_df.columns:
        actual_tariff_df[col] = None

merge_payload_cols = [c for c in required_tariff_cols if c != 'agr_id_key']
drop_before_merge = []
for c in merge_payload_cols:
    for c_try in [c, f'{c}_x', f'{c}_y']:
        if c_try in base_df.columns:
            drop_before_merge.append(c_try)
if drop_before_merge:
    base_df = base_df.drop(columns=sorted(set(drop_before_merge)))

base_df = base_df.merge(
    actual_tariff_df[required_tariff_cols],
    on='agr_id_key',
    how='left'
)

base_df['tariff_name'] = base_df['tariff_name_actual'].where(
    base_df['tariff_name_actual'].notna() & (base_df['tariff_name_actual'].astype(str).str.strip() != ''),
    base_df['tariff_name_legacy']
)
base_df['tariff_source'] = base_df['tariff_name_actual'].apply(
    lambda x: 'agr_id_actual' if pd.notna(x) and str(x).strip() else 'legacy_cft'
)
base_df['tariff_short'] = base_df['tariff_name'].apply(lambda x: build_tariff_short(x, source='lake'))

commission_monthly_actual_num = pd.to_numeric(base_df.get('commission_monthly_actual'), errors='coerce')
commission_monthly_legacy_num = pd.to_numeric(base_df.get('commission_monthly_legacy'), errors='coerce')
commission_monthly_plan_fallback_num = pd.Series(np.nan, index=base_df.index, dtype='float64')

if tariff_fix_map_df is not None and len(tariff_fix_map_df):
    tariff_fix_map_series = (
        tariff_fix_map_df.assign(c_tariff_plan=lambda d: d['c_tariff_plan'].astype(str).str.strip())
        .dropna(subset=['c_tariff_plan'])
        .drop_duplicates(subset=['c_tariff_plan'], keep='first')
        .set_index('c_tariff_plan')['commission_monthly_fix']
    )
    tariff_plan_for_fallback = base_df.get('c_tariff_plan', pd.Series(index=base_df.index, dtype=object))
    commission_monthly_plan_fallback_num = pd.to_numeric(
        tariff_plan_for_fallback.astype(str).str.strip().map(tariff_fix_map_series),
        errors='coerce'
    )

if use_legacy_commission_monthly_only:
    base_df['commission_monthly'] = commission_monthly_legacy_num
else:
    base_df['commission_monthly'] = commission_monthly_actual_num.where(
        commission_monthly_actual_num.notna(),
        commission_monthly_plan_fallback_num.where(
            commission_monthly_plan_fallback_num.notna(),
            commission_monthly_legacy_num
        )
    )

commission_actual_fill_pct = round(100.0 * commission_monthly_actual_num.notna().mean(), 2)
commission_plan_fill_pct = round(100.0 * commission_monthly_plan_fallback_num.notna().mean(), 2)
commission_legacy_fill_pct = round(100.0 * commission_monthly_legacy_num.notna().mean(), 2)
commission_zero_pct = round(100.0 * pd.to_numeric(base_df.get('commission_monthly'), errors='coerce').fillna(0).eq(0).mean(), 2)

commission_from_ops_num = pd.to_numeric(base_df.get('commission_from_ops'), errors='coerce').fillna(0)
commission_monthly_num = pd.to_numeric(base_df.get('commission_monthly'), errors='coerce').fillna(0)
int_component_num = pd.to_numeric(base_df.get('int_component'), errors='coerce').fillna(0)
amortization_num = pd.to_numeric(base_df.get('amortization'), errors='coerce').fillna(0)
retl_cnt_num = pd.to_numeric(base_df.get('retl_cnt'), errors='coerce').fillna(0)

base_df['commission_total'] = commission_from_ops_num + commission_monthly_num
base_df['aur'] = retl_cnt_num * 1926
base_df['chod'] = base_df['commission_total'] + int_component_num
base_df['fin_result'] = base_df['chod'] - pd.to_numeric(base_df['aur'], errors='coerce').fillna(0) - amortization_num
base_df['report_month'] = report_month_label
base_df['snapshot_month_start'] = snapshot_month_start

mvp_columns = [
    'report_month', 'snapshot_month_start', 'inn', 'company_name',
    'agr_id', 'n_agr', 'contract_number', 'd_valid_from', 'd_valid_to',
    'n_agr_actual', 'contract_number_acq', 'd_valid_from_actual', 'd_valid_to_actual',
    'cdi_id', 'ssp_ocrm', 'cft_id', 'ogrn', 'filial_rf', 'vsp_name', 'vsp_code',
    'tariff_name', 'tariff_short', 'tariff_source',
    'retl_cnt', 'term_cnt', 'active_terms', 'active_term_cnt', 'trx_cnt', 'trx_sum',
    'commission_from_ops', 'commission_monthly', 'int_component', 'commission_total',
    'aur', 'amortization', 'chod', 'fin_result'
]

for col in mvp_columns:
    if col not in base_df.columns:
        base_df[col] = None

for col in [
    'trx_sum', 'commission_from_ops', 'commission_monthly', 'int_component',
    'commission_total', 'aur', 'amortization', 'chod', 'fin_result'
]:
    base_df[col] = base_df[col].map(to_money_decimal_2_or_none)

for col in ['retl_cnt', 'term_cnt', 'active_terms', 'active_term_cnt', 'trx_cnt']:
    base_df[col] = pd.to_numeric(base_df[col], errors='coerce').astype('Int64')

final_df_before_coef = base_df[mvp_columns].copy()
final_df = final_df_before_coef.copy()

print(f'10_final_df rows: {len(final_df):,}')
print(
    f"commission fill rates: actual={commission_actual_fill_pct}%, "
    f"plan_fallback_08b={commission_plan_fill_pct}%, legacy={commission_legacy_fill_pct}%, "
    f"zero_final={commission_zero_pct}%"
)

## Секция 10b. Dashboard-даты договора (fallback)
Кратко: для дашборда заполняем даты договора по правилу `actual на month_end -> последний исторический SA-договор`, и считаем сколько строк заполнилось fallback-ом.

In [ ]:
# Dashboard contract dates:
# 1) берем actual-даты (активный договор на month_end)
# 2) если actual отсутствует -> берем последний исторический закрытый SA-договор по agr_id

agr_scope_for_history = sorted([
    x for x in final_df_before_coef.get('agr_id', pd.Series(dtype=object)).map(normalize_agr_q1).dropna().astype(str).str.strip().unique().tolist()
    if x
])

history_parts = []
if agr_scope_for_history:
    for agr_chunk in _split_scope(agr_scope_for_history, 1000):
        agr_in = _in_sql_list(agr_chunk)
        sql_last_hist_contract = f"""
        with hist_ranked as (
          select
            cast(a.abs_agr_id as string) as agr_id_key,
            cast(a.n_agr as string) as n_agr_last_hist,
            cast(a.c_agr_number as string) as contract_number_last_hist,
            cast(a.d_valid_from as date) as d_valid_from_last_hist,
            cast(a.d_valid_to as date) as d_valid_to_last_hist,
            row_number() over (
              partition by cast(a.abs_agr_id as string)
              order by cast(a.d_valid_to as date) desc,
                       cast(a.d_valid_from as date) desc,
                       cast(a.n_agr as string) desc
            ) as rn
          from ods_alpha.scd1_agreements a
          where cast(a.abs_agr_id as string) in ({agr_in})
            and upper(trim(cast(a.acq_class as string))) = 'SA'
            and a.d_valid_to is not null
            and cast(a.d_valid_to as date) <= cast('{month_end}' as date)
            and coalesce(a.ods_deleted_flg, '0') <> '1'
        )
        select
          agr_id_key,
          n_agr_last_hist,
          contract_number_last_hist,
          d_valid_from_last_hist,
          d_valid_to_last_hist
        from hist_ranked
        where rn = 1
        """
        part_df = _run_impala_fetch(imp, sql_last_hist_contract, mem_limit='8g')
        if part_df is not None and len(part_df):
            history_parts.append(part_df)

if history_parts:
    agr_history_close_df = pd.concat(history_parts, ignore_index=True)
    agr_history_close_df['agr_id_key'] = agr_history_close_df['agr_id_key'].astype(str).str.strip()
    agr_history_close_df = agr_history_close_df.drop_duplicates(subset=['agr_id_key'], keep='first')
else:
    agr_history_close_df = pd.DataFrame(columns=[
        'agr_id_key', 'n_agr_last_hist', 'contract_number_last_hist',
        'd_valid_from_last_hist', 'd_valid_to_last_hist'
    ])

final_df_before_coef['agr_id_key'] = final_df_before_coef['agr_id'].map(normalize_agr_q1)
final_df_before_coef = final_df_before_coef.merge(agr_history_close_df, on='agr_id_key', how='left')

final_df_before_coef['d_valid_from_dashboard'] = final_df_before_coef.get('d_valid_from_actual')
final_df_before_coef['d_valid_to_dashboard'] = final_df_before_coef.get('d_valid_to_actual')

fallback_mask_dashboard = final_df_before_coef['d_valid_from_dashboard'].isna()
final_df_before_coef.loc[fallback_mask_dashboard, 'd_valid_from_dashboard'] = final_df_before_coef.loc[fallback_mask_dashboard, 'd_valid_from_last_hist']
final_df_before_coef.loc[fallback_mask_dashboard, 'd_valid_to_dashboard'] = final_df_before_coef.loc[fallback_mask_dashboard, 'd_valid_to_last_hist']

final_df_before_coef['contract_dates_source_dashboard'] = np.where(
    final_df_before_coef['d_valid_from_actual'].notna(),
    'actual_month_end',
    np.where(
        final_df_before_coef['d_valid_from_last_hist'].notna(),
        'historical_fallback',
        'not_found'
    )
)

filled_actual_rows = int(final_df_before_coef['d_valid_from_actual'].notna().sum())
filled_dashboard_rows = int(final_df_before_coef['d_valid_from_dashboard'].notna().sum())
filled_by_fallback_rows = int((final_df_before_coef['contract_dates_source_dashboard'] == 'historical_fallback').sum())
not_found_rows = int((final_df_before_coef['contract_dates_source_dashboard'] == 'not_found').sum())

dashboard_dates_fill_compare_df = pd.DataFrame([
    {'metric': 'rows_total', 'value': int(len(final_df_before_coef))},
    {'metric': 'filled_actual_rows', 'value': filled_actual_rows},
    {'metric': 'filled_dashboard_rows_after_fallback', 'value': filled_dashboard_rows},
    {'metric': 'filled_by_historical_fallback_rows', 'value': filled_by_fallback_rows},
    {'metric': 'still_not_found_rows', 'value': not_found_rows},
])

print('Dashboard contract dates fill comparison:')
dashboard_dates_fill_compare_df

# Дальше работаем с обновленным final_df (до коэффициентов)
final_df = final_df_before_coef.copy()

In [ ]:
METRICS_ORDER = [
    'unique_inn',
    'retl_cnt',
    'term_cnt',
    'trx_cnt',
    'trx_sum',
    'commission_from_ops',
    'commission_monthly',
    'commission_total',
    'int_component',
    'chod',
]


def build_lake_agg(final_df):
    if final_df is None or final_df.empty:
        return pd.DataFrame(
            columns=[
                'inn_key', 'agr_id_key', 'retl_cnt_lake', 'term_cnt_lake', 'trx_cnt_lake',
                'trx_sum_lake', 'commission_from_ops_lake', 'commission_monthly_lake',
                'commission_total_lake', 'int_component_lake', 'chod_lake'
            ]
        )

    lk = final_df.copy()
    lk['inn_key'] = lk['inn'].apply(normalize_inn_q1)
    lk['agr_id_key'] = lk['agr_id'].apply(normalize_agr_q1)

    lk['retl_cnt_lake'] = pd.to_numeric(lk['retl_cnt'], errors='coerce')
    lk['term_cnt_lake'] = pd.to_numeric(lk['term_cnt'], errors='coerce')
    lk['trx_cnt_lake'] = pd.to_numeric(lk['trx_cnt'], errors='coerce')
    lk['trx_sum_lake'] = pd.to_numeric(lk['trx_sum'], errors='coerce')
    lk['commission_from_ops_lake'] = pd.to_numeric(lk['commission_from_ops'], errors='coerce')
    lk['commission_monthly_lake'] = pd.to_numeric(lk['commission_monthly'], errors='coerce')
    lk['commission_total_lake'] = pd.to_numeric(lk['commission_total'], errors='coerce')
    lk['int_component_lake'] = pd.to_numeric(lk['int_component'], errors='coerce')
    lk['chod_lake'] = pd.to_numeric(lk['chod'], errors='coerce')

    lake_agg = (
        lk.dropna(subset=['inn_key', 'agr_id_key'])
          .groupby(['inn_key', 'agr_id_key'], as_index=False)
          .agg({
              'retl_cnt_lake': 'max',
              'term_cnt_lake': 'max',
              'trx_cnt_lake': 'max',
              'trx_sum_lake': 'max',
              'commission_from_ops_lake': 'max',
              'commission_monthly_lake': 'max',
              'commission_total_lake': 'max',
              'int_component_lake': 'max',
              'chod_lake': 'max',
          })
    )
    return lake_agg


def calc_lake_totals(lake_agg):
    if lake_agg is None or lake_agg.empty:
        return {m: 0.0 if m != 'unique_inn' else 0 for m in METRICS_ORDER}

    return {
        'unique_inn': int(lake_agg['inn_key'].nunique()),
        'retl_cnt': float(lake_agg['retl_cnt_lake'].fillna(0).sum()),
        'term_cnt': float(lake_agg['term_cnt_lake'].fillna(0).sum()),
        'trx_cnt': float(lake_agg['trx_cnt_lake'].fillna(0).sum()),
        'trx_sum': float(lake_agg['trx_sum_lake'].fillna(0).sum()),
        'commission_from_ops': float(lake_agg['commission_from_ops_lake'].fillna(0).sum()),
        'commission_monthly': float(lake_agg['commission_monthly_lake'].fillna(0).sum()),
        'commission_total': float(lake_agg['commission_total_lake'].fillna(0).sum()),
        'int_component': float(lake_agg['int_component_lake'].fillna(0).sum()),
        'chod': float(lake_agg['chod_lake'].fillna(0).sum()),
    }


lake_agg_before_coef = build_lake_agg(final_df_before_coef)
lake_totals_before_coef = calc_lake_totals(lake_agg_before_coef)

summary_metrics_df_before_coef = pd.DataFrame(
    [{'metric': m, 'value': lake_totals_before_coef[m]} for m in METRICS_ORDER]
)

print('Итоговая таблица core-метрик (до коэффициентов, scoped-методика):')
summary_metrics_df_before_coef

In [ ]:
# Проверка паритета lake_value-логики для текущего месяца (до коэффициентов)

parity_check_before_coef = {
    'report_month': report_month_label,
    'final_df_rows': int(len(final_df_before_coef)),
    'lake_agg_rows': int(len(lake_agg_before_coef)),
    'unique_inn': lake_totals_before_coef['unique_inn'],
    'commission_monthly': lake_totals_before_coef['commission_monthly'],
    'commission_total': lake_totals_before_coef['commission_total'],
    'fin_result_raw_sum_reference': float(pd.to_numeric(final_df_before_coef.get('fin_result'), errors='coerce').fillna(0).sum()),
}

print('Parity check (scoped aggregation for core metrics):')
print(parity_check_before_coef)

## Проверка заполненности d_valid_from_actual
Кратко: диагностируем долю пустых значений, разбивку по `agr_id_source` и наличие активного SA-договора на `month_end`.

In [ ]:
# Диагностика пустых d_valid_from_actual

d_valid_diag_df = final_df_before_coef[['agr_id', 'd_valid_from_actual']].copy()
d_valid_diag_df['agr_id_key'] = d_valid_diag_df['agr_id'].map(normalize_agr_q1)
d_valid_diag_df['d_valid_from_actual_is_null'] = d_valid_diag_df['d_valid_from_actual'].isna()

if 'agr_id_key' in base_df.columns and 'agr_id_source' in base_df.columns:
    agr_source_map_df = (
        base_df[['agr_id_key', 'agr_id_source']]
        .dropna(subset=['agr_id_key'])
        .drop_duplicates(subset=['agr_id_key'], keep='first')
    )
    d_valid_diag_df = d_valid_diag_df.merge(agr_source_map_df, on='agr_id_key', how='left')
else:
    d_valid_diag_df['agr_id_source'] = 'unknown'

d_valid_diag_df['agr_id_source'] = d_valid_diag_df['agr_id_source'].fillna('unknown')

total_rows = int(len(d_valid_diag_df))
null_rows = int(d_valid_diag_df['d_valid_from_actual_is_null'].sum())
null_pct = round((null_rows / total_rows) * 100.0, 2) if total_rows else 0.0

null_by_source_df = (
    d_valid_diag_df.groupby('agr_id_source', as_index=False)
    .agg(
        rows=('agr_id_source', 'size'),
        null_rows=('d_valid_from_actual_is_null', 'sum')
    )
)
null_by_source_df['null_pct'] = np.where(
    null_by_source_df['rows'] > 0,
    (null_by_source_df['null_rows'] / null_by_source_df['rows'] * 100.0).round(2),
    0.0
)

null_agr_scope = sorted([
    x for x in d_valid_diag_df.loc[
        d_valid_diag_df['d_valid_from_actual_is_null'], 'agr_id_key'
    ].dropna().astype(str).str.strip().unique().tolist()
    if x
])

agreements_presence_parts = []
if null_agr_scope:
    scope_chunks = _split_scope(null_agr_scope, 1000)
    for agr_chunk in scope_chunks:
        agr_in = _in_sql_list(agr_chunk)
        sql_check_active_sa = f"""
        select
          cast(a.abs_agr_id as string) as agr_id_key,
          count(*) as matched_rows
        from ods_alpha.scd1_agreements a
        where cast(a.abs_agr_id as string) in ({agr_in})
          and upper(trim(cast(a.acq_class as string))) = 'SA'
          and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
          and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_end}' as date))
          and coalesce(a.ods_deleted_flg, '0') <> '1'
        group by cast(a.abs_agr_id as string)
        """
        part_df = _run_impala_fetch(imp, sql_check_active_sa, mem_limit='8g')
        if part_df is not None and len(part_df):
            agreements_presence_parts.append(part_df)

if agreements_presence_parts:
    agreements_presence_df = pd.concat(agreements_presence_parts, ignore_index=True)
    agreements_presence_df['agr_id_key'] = agreements_presence_df['agr_id_key'].astype(str).str.strip()
    agreements_presence_df['matched_rows'] = pd.to_numeric(agreements_presence_df['matched_rows'], errors='coerce').fillna(0)
    agreements_presence_df = agreements_presence_df.groupby('agr_id_key', as_index=False)['matched_rows'].sum()
else:
    agreements_presence_df = pd.DataFrame(columns=['agr_id_key', 'matched_rows'])

null_agr_diag_df = pd.DataFrame({'agr_id_key': null_agr_scope})
null_agr_diag_df = null_agr_diag_df.merge(agreements_presence_df, on='agr_id_key', how='left')
null_agr_diag_df['matched_rows'] = pd.to_numeric(null_agr_diag_df['matched_rows'], errors='coerce').fillna(0)
null_agr_diag_df['has_active_sa_agreement_month_end'] = null_agr_diag_df['matched_rows'] > 0

agreement_presence_summary_df = (
    null_agr_diag_df.groupby('has_active_sa_agreement_month_end', as_index=False)
    .agg(agr_id_cnt=('agr_id_key', 'size'))
)

print(f"[{report_month_label}] d_valid_from_actual null rows: {null_rows:,} / {total_rows:,} ({null_pct}%)")
print(f"[{report_month_label}] unique agr_id among nulls: {len(null_agr_scope):,}")

print('Null rate by agr_id_source:')
null_by_source_df

print('Active SA agreement presence for null agr_id at month_end:')
agreement_presence_summary_df

## Секция 10c. Сравнение дат договора и тарифа с Excel (апрель)
Кратко: сравниваем `d_valid_from`, `d_valid_to`, `tariff_name` из `final_df_before_coef` с Excel по ключу `agr_id + Номер договора`.

In [ ]:
def pick_col_robust(columns, candidates):
    cols = list(columns)

    def _norm_col_name(s):
        return re.sub(r'\s+', ' ', str(s).replace('\xa0', ' ').strip().lower())

    norm_map = {_norm_col_name(c): c for c in cols}
    for c in candidates:
        if c in cols:
            return c
        nc = _norm_col_name(c)
        if nc in norm_map:
            return norm_map[nc]
    return None


def _norm_contract_key(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '')
    s = re.sub(r'\.0$', '', s)
    return s if s else None


def _norm_tariff_key(v):
    if pd.isna(v):
        return None
    s = str(v).replace('\xa0', ' ')
    s = re.sub(r'\s+', ' ', s).strip().lower()
    return s or None


excel_april_df_raw = pd.read_excel(excel_april_compare_path, header=excel_april_compare_header)

excel_col_map = {
    'agr_id_col': ['ID договора', 'agr_id', 'abs_agr_id'],
    'contract_number_col': ['Номер договора', 'contract_number', 'Номер договора (acq)'],
    'd_valid_from_col': ['Дата регистрации договора', 'Дата начала договора', 'd_valid_from', 'Дата начала'],
    'd_valid_to_col': ['Дата закрытия договора', 'Дата окончания договора', 'd_valid_to', 'Дата окончания'],
    'tariff_col': ['Тариф', 'Тарифный план', 'tariff_name'],
}
resolved_excel_cols = {k: pick_col_robust(excel_april_df_raw.columns, v) for k, v in excel_col_map.items()}
missing_excel_cols = [k for k, v in resolved_excel_cols.items() if v is None]
if missing_excel_cols:
    raise ValueError(f'Excel compare: не найдены колонки {missing_excel_cols}. Доступные: {list(excel_april_df_raw.columns)}')

compare_final_df = final_df_before_coef.copy()
compare_final_df['agr_id_key'] = compare_final_df['agr_id'].map(normalize_agr_q1)
compare_final_df['contract_number_key'] = compare_final_df['contract_number'].map(_norm_contract_key)
compare_final_df['d_valid_from_lake'] = pd.to_datetime(compare_final_df['d_valid_from'], errors='coerce').dt.date
compare_final_df['d_valid_to_lake'] = pd.to_datetime(compare_final_df['d_valid_to'], errors='coerce').dt.date
compare_final_df['tariff_name_lake'] = compare_final_df['tariff_name'].apply(_norm_tariff_key)

compare_final_df = (
    compare_final_df[
        ['agr_id_key', 'contract_number_key', 'd_valid_from_lake', 'd_valid_to_lake', 'tariff_name_lake']
    ]
    .dropna(subset=['agr_id_key', 'contract_number_key'])
    .drop_duplicates(subset=['agr_id_key', 'contract_number_key'], keep='first')
)

compare_excel_df = excel_april_df_raw.copy()
compare_excel_df['agr_id_key'] = compare_excel_df[resolved_excel_cols['agr_id_col']].map(normalize_agr_q1)
compare_excel_df['contract_number_key'] = compare_excel_df[resolved_excel_cols['contract_number_col']].map(_norm_contract_key)
compare_excel_df['d_valid_from_excel'] = pd.to_datetime(compare_excel_df[resolved_excel_cols['d_valid_from_col']], errors='coerce').dt.date
compare_excel_df['d_valid_to_excel'] = pd.to_datetime(compare_excel_df[resolved_excel_cols['d_valid_to_col']], errors='coerce').dt.date
compare_excel_df['tariff_name_excel'] = compare_excel_df[resolved_excel_cols['tariff_col']].apply(_norm_tariff_key)

compare_excel_df = (
    compare_excel_df[
        ['agr_id_key', 'contract_number_key', 'd_valid_from_excel', 'd_valid_to_excel', 'tariff_name_excel']
    ]
    .dropna(subset=['agr_id_key', 'contract_number_key'])
    .drop_duplicates(subset=['agr_id_key', 'contract_number_key'], keep='first')
)

april_compare_df = compare_final_df.merge(
    compare_excel_df,
    on=['agr_id_key', 'contract_number_key'],
    how='outer',
    indicator=True
)

april_compare_df['d_valid_from_match'] = (
    april_compare_df['d_valid_from_lake'].astype(str).fillna('None') ==
    april_compare_df['d_valid_from_excel'].astype(str).fillna('None')
)
april_compare_df['d_valid_to_match'] = (
    april_compare_df['d_valid_to_lake'].astype(str).fillna('None') ==
    april_compare_df['d_valid_to_excel'].astype(str).fillna('None')
)
april_compare_df['tariff_name_match'] = (
    april_compare_df['tariff_name_lake'].fillna('') == april_compare_df['tariff_name_excel'].fillna('')
)
april_compare_df['all_3_match'] = (
    april_compare_df['d_valid_from_match'] &
    april_compare_df['d_valid_to_match'] &
    april_compare_df['tariff_name_match']
)

rows_in_final_df = int(len(compare_final_df))
rows_in_excel = int(len(compare_excel_df))
rows_matched_by_key = int((april_compare_df['_merge'] == 'both').sum())
rows_only_final_df = int((april_compare_df['_merge'] == 'left_only').sum())
rows_only_excel = int((april_compare_df['_merge'] == 'right_only').sum())

both_mask = april_compare_df['_merge'] == 'both'
matched_scope_cnt = int(both_mask.sum())

april_compare_summary_df = pd.DataFrame([
    {'metric': 'rows_in_final_df', 'value': rows_in_final_df},
    {'metric': 'rows_in_excel', 'value': rows_in_excel},
    {'metric': 'rows_matched_by_key', 'value': rows_matched_by_key},
    {'metric': 'rows_only_in_final_df', 'value': rows_only_final_df},
    {'metric': 'rows_only_in_excel', 'value': rows_only_excel},
    {
        'metric': 'd_valid_from_match_rows',
        'value': int((both_mask & april_compare_df['d_valid_from_match']).sum())
    },
    {
        'metric': 'd_valid_to_match_rows',
        'value': int((both_mask & april_compare_df['d_valid_to_match']).sum())
    },
    {
        'metric': 'tariff_name_match_rows',
        'value': int((both_mask & april_compare_df['tariff_name_match']).sum())
    },
    {
        'metric': 'all_3_match_rows',
        'value': int((both_mask & april_compare_df['all_3_match']).sum())
    },
    {
        'metric': 'all_3_match_pct_of_matched',
        'value': round(100.0 * int((both_mask & april_compare_df['all_3_match']).sum()) / matched_scope_cnt, 2) if matched_scope_cnt else 0.0
    },
])

mismatch_detail_df = april_compare_df[
    (~april_compare_df['all_3_match']) |
    (april_compare_df['_merge'] != 'both')
][[
    'agr_id_key', 'contract_number_key', '_merge',
    'd_valid_from_lake', 'd_valid_from_excel', 'd_valid_from_match',
    'd_valid_to_lake', 'd_valid_to_excel', 'd_valid_to_match',
    'tariff_name_lake', 'tariff_name_excel', 'tariff_name_match',
    'all_3_match'
]].copy()

mismatch_detail_df = mismatch_detail_df.head(200).reset_index(drop=True)

print('April compare summary (d_valid_from / d_valid_to / tariff_name):')
april_compare_summary_df

In [ ]:
print('Mismatch detail sample (top 200):')
mismatch_detail_df

## Секция 11. Применение апрельских коэффициентов
Кратко: загружаем коэффициенты по `agr_id`, создаем доп. колонки и пересчитываем производные метрики.

In [ ]:
coef_df = pd.read_csv(coef_csv_path)

required_coef_cols = {'agr_id_key', 'k_comm_monthly_agr'}
missing_coef_cols = required_coef_cols - set(coef_df.columns)
if missing_coef_cols:
    raise ValueError(f'В файле коэффициентов отсутствуют колонки: {missing_coef_cols}. Доступные: {list(coef_df.columns)}')

coef_df['agr_id_key'] = coef_df['agr_id_key'].apply(normalize_agr_q1)
coef_df['k_comm_monthly_agr'] = pd.to_numeric(coef_df['k_comm_monthly_agr'], errors='coerce')

if 'coef_source' not in coef_df.columns:
    coef_df['coef_source'] = 'agr_ratio'

coef_df = coef_df[['agr_id_key', 'k_comm_monthly_agr', 'coef_source']].drop_duplicates(subset=['agr_id_key'], keep='first')

final_df['agr_id_key'] = final_df['agr_id'].apply(normalize_agr_q1)
final_df['commission_monthly_lake'] = pd.to_numeric(final_df['commission_monthly'], errors='coerce').fillna(0)

final_df = final_df.merge(coef_df, on='agr_id_key', how='left')

median_k = float(coef_df['k_comm_monthly_agr'].dropna().median()) if coef_df['k_comm_monthly_agr'].notna().any() else 1.0
missing_k_mask = final_df['k_comm_monthly_agr'].isna()
final_df.loc[missing_k_mask, 'k_comm_monthly_agr'] = median_k
final_df.loc[missing_k_mask, 'coef_source'] = 'fallback'
final_df['coef_source'] = final_df['coef_source'].fillna('agr_ratio')

final_df['com_forcast'] = final_df['commission_monthly_lake'] * final_df['k_comm_monthly_agr']
final_df['commission_monthly'] = final_df['com_forcast']

print(f'Коэффициенты загружены: {len(coef_df):,}')
print(f'Median k для fallback: {median_k:,.6f}')
print(f'Строк с fallback коэффициента: {int(missing_k_mask.sum()):,}')

In [ ]:
commission_from_ops_num = pd.to_numeric(final_df.get('commission_from_ops'), errors='coerce').fillna(0)
commission_monthly_num = pd.to_numeric(final_df.get('commission_monthly'), errors='coerce').fillna(0)
int_component_num = pd.to_numeric(final_df.get('int_component'), errors='coerce').fillna(0)
amortization_num = pd.to_numeric(final_df.get('amortization'), errors='coerce').fillna(0)
aur_num = pd.to_numeric(final_df.get('aur'), errors='coerce').fillna(0)

final_df['commission_total'] = commission_from_ops_num + commission_monthly_num
final_df['chod'] = final_df['commission_total'] + int_component_num
final_df['fin_result'] = final_df['chod'] - aur_num - amortization_num

print('Контрольные суммы после корректировки commission_monthly:')
print(f"commission_monthly_lake_total: {final_df['commission_monthly_lake'].sum():,.2f}")
print(f"commission_monthly_forecast_total: {final_df['com_forcast'].sum():,.2f}")
print(f"commission_total_total: {pd.to_numeric(final_df['commission_total'], errors='coerce').fillna(0).sum():,.2f}")

In [ ]:
def _safe_sum(df, col):
    return pd.to_numeric(df.get(col), errors='coerce').fillna(0).sum()

parity_before = {
    'rows': len(final_df_before_coef),
    'commission_monthly_sum': _safe_sum(final_df_before_coef, 'commission_monthly'),
    'commission_total_sum': _safe_sum(final_df_before_coef, 'commission_total'),
    'fin_result_sum': _safe_sum(final_df_before_coef, 'fin_result'),
    'tariff_name_fill_pct': (final_df_before_coef['tariff_name'].astype(str).str.strip().ne('').mean() * 100.0) if len(final_df_before_coef) else 0.0,
    'commission_monthly_fill_pct': (pd.to_numeric(final_df_before_coef['commission_monthly'], errors='coerce').notna().mean() * 100.0) if len(final_df_before_coef) else 0.0,
}

parity_after = {
    'rows': len(final_df),
    'commission_monthly_sum': _safe_sum(final_df, 'commission_monthly'),
    'commission_total_sum': _safe_sum(final_df, 'commission_total'),
    'fin_result_sum': _safe_sum(final_df, 'fin_result'),
}

required_cols = ['commission_monthly_lake', 'k_comm_monthly_agr', 'commission_monthly', 'com_forcast', 'coef_source']
missing_cols = [c for c in required_cols if c not in final_df.columns]
if missing_cols:
    raise ValueError(f'В final_df отсутствуют обязательные колонки: {missing_cols}')

print('Паритет до применения коэффициентов (базовая scoped-логика):')
print(parity_before)
print('Итог после коэффициентов:')
print(parity_after)
print(f"Доля fallback по коэффициентам: {(final_df['coef_source'] == 'fallback').mean() * 100:.2f}%")

## Секция 12. Экспорт
Кратко: сохраняем итоговый CSV для передачи коллеге.

In [ ]:
if save_csv:
    output_path = Path(output_csv_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    export_columns = [
        'report_month', 'snapshot_month_start', 'inn', 'company_name',
        'agr_id', 'n_agr', 'contract_number', 'd_valid_from', 'd_valid_to',
        'n_agr_actual', 'contract_number_acq', 'd_valid_from_actual', 'd_valid_to_actual',
        'd_valid_from_dashboard', 'd_valid_to_dashboard', 'contract_dates_source_dashboard',
        'cdi_id', 'ssp_ocrm', 'cft_id', 'ogrn', 'filial_rf', 'vsp_name', 'vsp_code',
        'tariff_name', 'tariff_short', 'tariff_source',
        'retl_cnt', 'term_cnt', 'active_terms', 'active_term_cnt', 'trx_cnt', 'trx_sum',
        'commission_from_ops', 'commission_monthly_lake', 'k_comm_monthly_agr', 'com_forcast',
        'commission_monthly', 'commission_total', 'int_component', 'amortization', 'aur', 'chod', 'fin_result',
        'coef_source'
    ]
    export_columns = [c for c in export_columns if c in final_df.columns]

    final_df[export_columns].to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f'CSV сохранен: {output_path.resolve()}')
    print(f'Экспортировано колонок: {len(export_columns)}')
else:
    print('Экспорт в CSV отключен (save_csv=False).')